# Experiment 2: Max Features Tuning
In this notebook, we investigate the impact of the `max_features` parameter in our text vectorization process.

### What is `max_features`?
The `max_features` parameter in `TfidfVectorizer` or `CountVectorizer` limits the vocabulary size by only considering the top N terms ordered by term frequency across the corpus.

### Why tune it?
- **Dimensionality Reduction**: Large vocabularies create very sparse and high-dimensional matrices, which can lead to the "Curse of Dimensionality".
- **Overfitting**: Rare words might capture noise rather than general patterns.
- **Computation**: Reducing features speeds up model training and inference.

In [4]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Max Features Tuning")


<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1776065148165, experiment_id='5', last_update_time=1776065148165, lifecycle_stage='active', name='Max Features Tuning', tags={}, workspace='default'>

In [5]:
# Load and preprocess
data = pd.read_csv(r'../data/processed/dataset.csv')
data = data.dropna(subset=['text_processed', 'sentiment']).drop_duplicates()

X_text = data['text_processed']
X_numeric = data[['char_count', 'word_count', 'avg_word_len']].values
y = data['sentiment']

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_numeric, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)

In [6]:
feature_counts = [500, 1000, 2500, 5000, 7500, 10000, None]

for max_f in feature_counts:
    with mlflow.start_run(run_name=f"TFIDF_max_features_{max_f}"):
        tfidf = TfidfVectorizer(max_features=max_f, ngram_range=(1,1))
        X_text_train_vec = tfidf.fit_transform(X_text_train)
        X_text_test_vec = tfidf.transform(X_text_test)
        
        X_train_final = hstack([X_text_train_vec, X_num_train_scaled])
        X_test_final = hstack([X_text_test_vec, X_num_test_scaled])
        
        # Train
        model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
        model.fit(X_train_final, y_train) # type: ignore
        y_pred = model.predict(X_test_final)
        
        # ── Log Parameters ────────────────────────────────────────────────────────
        mlflow.log_params({
                   # Split
            "test_size"             : 0.2,
            "stratify"              : True,
            "random_state"          : 42,

            # Feature Representation
            "representation"        : f"TFIDF_{max_f}",
            "vectorizer_type"       : "TfidfVectorizer",
            "tfidf_max_features"    : max_f,
            "ngram_range"           : "(1,1)",

            # Scaling
            "scaler"                : "StandardScaler",
            "scaled_features"       : "char_count, word_count, avg_word_len",

            # Model
            "model"                 : "LogisticRegression",
            "max_iter"              : 1000,
            "solver"                : "lbfgs",
            "random_state"          : 42,
            "class_weight"          : "balanced",

            # Features
            "num_custom_features"   : 3,
            "text_features_count"   : X_text_train_vec.shape[1],
        })

        # Metrics Calculation
        report_dict:dict = classification_report(y_test, y_pred, output_dict=True) # type: ignore
        
        metrics = {
            "accuracy"         : accuracy_score(y_test, y_pred),
            "f1_weighted"      : report_dict['weighted avg']['f1-score'],
            "f1_macro"         : report_dict['macro avg']['f1-score'],
            "precision_weighted": report_dict['weighted avg']['precision'],
            "precision_macro"  : report_dict['macro avg']['precision'],
            "recall_weighted"  : report_dict['weighted avg']['recall'],
            "recall_macro"     : report_dict['macro avg']['recall'],
        }
        
        # Log Per-Class Metrics
        for label in model.classes_:
            metrics[f"f1_class_{label}"] = report_dict[str(label)]['f1-score']
            metrics[f"precision_class_{label}"] = report_dict[str(label)]['precision']
            metrics[f"recall_class_{label}"] = report_dict[str(label)]['recall']
            
        mlflow.log_metrics(metrics)
        
        print(f"Max Features: {max_f} -> F1 (Weighted): {metrics['f1_weighted']:.4f}")


Max Features: 500 -> F1 (Weighted): 0.6724
🏃 View run TFIDF_max_features_500 at: http://localhost:5000/#/experiments/5/runs/20d2115954b34f1d90813f936813f045
🧪 View experiment at: http://localhost:5000/#/experiments/5
Max Features: 1000 -> F1 (Weighted): 0.6815
🏃 View run TFIDF_max_features_1000 at: http://localhost:5000/#/experiments/5/runs/c1255355ac054da7b7423d25a888faf1
🧪 View experiment at: http://localhost:5000/#/experiments/5
Max Features: 2500 -> F1 (Weighted): 0.7005
🏃 View run TFIDF_max_features_2500 at: http://localhost:5000/#/experiments/5/runs/288ed454bce64a5880612b2bc7d94760
🧪 View experiment at: http://localhost:5000/#/experiments/5
Max Features: 5000 -> F1 (Weighted): 0.7056
🏃 View run TFIDF_max_features_5000 at: http://localhost:5000/#/experiments/5/runs/e01d567d7db04d42ae58416710d9c867
🧪 View experiment at: http://localhost:5000/#/experiments/5
Max Features: 7500 -> F1 (Weighted): 0.6979
🏃 View run TFIDF_max_features_7500 at: http://localhost:5000/#/experiments/5/runs/

#### Conclusion
TFID with unigram (1, 1) with 500 features is giving better scores of recall, precision.
although TFID 7000 is optimal than others but with 500 there is very little comparable with 7000 and lesser features with 500 can decrease model training time and reduce curse of dimn and reduce overfitting (low bais high varaince model). 